# 🧪 Module 01: Exploratory Data Analysis & Sample Ratio Mismatch (SRM) Audit
**Project**: A/B Test Analysis — Marketing Campaign & Landing Page Conversion  
**Dataset**: Public E-Commerce / Marketing Campaign A/B Test Dataset  
**Author**: Data Analytics Portfolio Project

---
### 📌 Objectives:
1. Load raw campaign telemetry records and perform data quality / integrity audits.
2. Filter routing mismatches and deduplicate user interactions (first-exposure rule).
3. Validate experimental randomization using a **Chi-Square Goodness-of-Fit** test for Sample Ratio Mismatch (SRM).
4. Summarize baseline descriptive statistics: Control vs Treatment group sizes, baseline Conversion Rate (CR), Click-Through Rate (CTR), and Average Revenue Per User (ARPU).


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import generate_ab_test_dataset, clean_ab_test_data
from src.statistical_tests import check_sample_ratio_mismatch

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Ingest Raw Dataset & Inspect Telemetry

In [ ]:
# Generate or load raw A/B test dataset
raw_df = pd.read_csv("../data/raw/marketing_ab_test_raw.csv")
print(f"Total raw observations: {len(raw_df):,}")
raw_df.head()


In [ ]:
# Data schema and missingness check
raw_df.info()


## 2. Data Cleaning & Integrity Audit
We audit for:
1. **Routing Inconsistencies**: Instances where `control` users received `new_landing_page` or `treatment` received `old_landing_page`.
2. **User Deduplication**: Users who revisited the site multiple times during the 30-day window (standard industry practice is keeping first valid exposure).


In [ ]:
cleaned_df = clean_ab_test_data(raw_df, output_path="../data/processed/ab_test_cleaned.csv")
cleaned_df.head()


## 3. Sample Ratio Mismatch (SRM) Diagnostic Test
A Sample Ratio Mismatch occurs when the ratio of users in control vs treatment differs significantly from the planned 50:50 allocation.
- **Null Hypothesis ($H_0$)**: The observed traffic allocation follows the expected 1:1 distribution ($\chi^2$ goodness-of-fit).
- **Alternate Hypothesis ($H_1$)**: There is a severe allocation bias / sample ratio mismatch.
- **Threshold**: $\alpha = 0.01$ (conservative to avoid false alarms).


In [ ]:
srm_results = check_sample_ratio_mismatch(cleaned_df, expected_ratio=(0.5, 0.5))

for k, v in srm_results.items():
    print(f"{k}: {v}")


## 4. Group Level KPI Summary
Comparison of Control vs Treatment across:
- Unique Visitors ($N$)
- Converters & Conversion Rate (%)
- Total Orders Placed
- Total Revenue & Average Revenue Per User (ARPU)
- Average Session Duration (seconds)


In [ ]:
summary = cleaned_df.groupby("group").agg(
    unique_visitors=("user_id", "count"),
    converters=("converted", "sum"),
    conversion_rate=("converted", "mean"),
    total_orders=("orders_count", "sum"),
    total_revenue=("revenue", "sum"),
    arpu=("revenue", "mean"),
    avg_session_duration=("session_duration_sec", "mean")
).reset_index()

summary["conversion_rate_pct"] = summary["conversion_rate"] * 100
summary
